# What is a British racecourse?

Study 03 separates **racecourse identity**, **stable physical course/track identity**, and lower-level **route/configuration/characteristic**. The 60 evidence notebooks under `racecourses/` are the source of truth for the national consolidation. One notebook covers both official Newmarket racecourses, so the notebook count is not itself a racecourse count. `candidate_course_label` remains a source classification rather than a governed venue or track ID.

In [ ]:
from pathlib import Path
import ast
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'studies').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RACECOURSE_DIR = PROJECT_ROOT / 'studies' / 'jurisdictions' / 'great_britain' / 'racecourses'
assert RACECOURSE_DIR.exists(), f'Racecourse notebook directory not found: {RACECOURSE_DIR}'
racecourse_notebooks = sorted(RACECOURSE_DIR.glob('*.ipynb'))
assert len(racecourse_notebooks) == 60, f'Expected 60 evidence notebooks, found {len(racecourse_notebooks)}'
print('racecourse evidence notebooks:', len(racecourse_notebooks))

In [ ]:
def assigned_names(source):
    tree = ast.parse(source)
    names = set()
    for node in ast.walk(tree):
        if isinstance(node, (ast.Assign, ast.AnnAssign)):
            targets = node.targets if isinstance(node, ast.Assign) else [node.target]
            for target in targets:
                if isinstance(target, ast.Name):
                    names.add(target.id)
    return names

def extract_dataframe(notebook_path, variable_name, *, required=True):
    notebook = json.loads(notebook_path.read_text())
    for cell in notebook.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        source = ''.join(cell.get('source', []))
        try:
            names = assigned_names(source)
        except SyntaxError:
            continue
        if variable_name not in names:
            continue
        namespace = {'pd': pd}
        exec(source, namespace)
        value = namespace.get(variable_name)
        if isinstance(value, pd.DataFrame):
            return value.copy()
    if required:
        raise RuntimeError(f'{variable_name!r} not found as a DataFrame in {notebook_path.name}')
    return pd.DataFrame()

## Racecourse identity and source labels

In [ ]:
source_label_mapping = pd.concat(
    [extract_dataframe(path, 'source_label_mapping') for path in racecourse_notebooks],
    ignore_index=True, sort=False,
)

print('source labels:', source_label_mapping['candidate_course_label'].nunique())
print('governed racecourse identities:', source_label_mapping['racecourse_identity'].nunique())

assert source_label_mapping['candidate_course_label'].nunique() == 65
assert source_label_mapping['racecourse_identity'].nunique() == 61

newmarket_mapping = source_label_mapping[
    source_label_mapping['candidate_course_label'].isin(['Newmarket', 'Newmarket (July)'])
][['candidate_course_label', 'racecourse_identity', 'racecourse_resolution_method']]
display(newmarket_mapping)

The 65 British source labels consolidate to **61 governed racecourse identities**. The important correction is Newmarket: official Jockey Club evidence identifies the **Rowley Mile** and **July Course** as two racecourses, not two tracks beneath one racecourse. Accordingly `Newmarket` resolves to **Newmarket — Rowley Mile** by the documented Source Version 1 label convention, while `Newmarket (July)` resolves explicitly to **Newmarket — July Course**. Other paired source labels such as Kempton/Kempton (AW), Lingfield/Lingfield (AW), Newcastle/Newcastle (AW), and Southwell/Southwell (AW) continue to map to one racecourse because their distinction is source classification rather than a separate official racecourse identity.

## Course/track inventory

In [ ]:
course_inventory = pd.concat(
    [extract_dataframe(path, 'course_inventory') for path in racecourse_notebooks],
    ignore_index=True, sort=False,
)

print('racecourses represented:', course_inventory['racecourse_identity'].nunique())
print('course/track inventory records:', len(course_inventory))

assert course_inventory['racecourse_identity'].nunique() == 61
assert len(course_inventory) == 90

## Stable course/track identities

Inventory rows are not automatically stable identities. Southwell's Fibresand/Tapeta rows are successive surface states of one all-weather track; Newcastle's former turf Flat track and Tapeta track are successive states of one persistent Flat-track identity; Windsor's dated Jump rows are temporary configurations of its turf course. These are the only national consolidation collapses applied here.

In [ ]:
stable_course_inventory = course_inventory.copy()
stable_course_inventory['stable_course_identity'] = stable_course_inventory['course_or_track_name']

resolved_collapses = {
    ('Southwell', 'All-Weather Flat Track — Fibresand'): 'All-Weather Flat Track',
    ('Southwell', 'All-Weather Flat Track — Tapeta'): 'All-Weather Flat Track',
    ('Newcastle', 'Former Flat Turf Track'): 'Flat Track',
    ('Newcastle', 'All-Weather Tapeta Track'): 'Flat Track',
    ('Windsor', 'Traditional Figure-of-Eight Turf Course'): 'Windsor Turf Course',
    ('Windsor', '2024/25 Jump Extended Left-Hand Oval'): 'Windsor Turf Course',
    ('Windsor', '2025/26 Jump Figure-of-Eight Configuration'): 'Windsor Turf Course',
}
for (racecourse, raw_name), stable_name in resolved_collapses.items():
    mask = (
        stable_course_inventory['racecourse_identity'].eq(racecourse)
        & stable_course_inventory['course_or_track_name'].eq(raw_name)
    )
    stable_course_inventory.loc[mask, 'stable_course_identity'] = stable_name

stable_identities = (
    stable_course_inventory[['racecourse_identity', 'stable_course_identity']]
    .drop_duplicates()
    .sort_values(['racecourse_identity', 'stable_course_identity'])
    .reset_index(drop=True)
)
identity_counts = (
    stable_identities.groupby('racecourse_identity').size()
    .rename('stable_course_identities').reset_index()
)

print('inventory records:', len(course_inventory))
print('stable course/track identities:', len(stable_identities))
print('racecourses with multiple stable identities:', (identity_counts['stable_course_identities'] > 1).sum())
print(identity_counts['stable_course_identities'].value_counts().sort_index())

assert len(stable_identities) == 86
assert (identity_counts['stable_course_identities'] > 1).sum() == 19


### Closeout correction

The national closeout now distinguishes **60 evidence notebooks** from **61 racecourse identities**. Newmarket is the reason: its single evidence notebook covers the two officially separate racecourses, Rowley Mile and July Course. This corrects the earlier analytical-parent treatment without changing the evidence that established the two-course hierarchy. Carlisle remains four peer course identities; Ayr remains Flat and Jumps with the six-furlong Straight Course below the peer layer.

## Remaining unresolved venue questions

In [ ]:
unresolved_tables = []
for path in racecourse_notebooks:
    table = extract_dataframe(path, 'unresolved_questions', required=False)
    if len(table):
        table = table.copy()
        table['source_notebook'] = path.name
        unresolved_tables.append(table)

study03_unresolved = (
    pd.concat(unresolved_tables, ignore_index=True, sort=False)
    if unresolved_tables else pd.DataFrame()
)
print('unresolved records:', len(study03_unresolved))
if len(study03_unresolved):
    print('racecourses with unresolved records:', study03_unresolved['racecourse_identity'].nunique())
    display(study03_unresolved)
assert len(study03_unresolved) == 7

No remaining unresolved item changes the governed national racecourse or peer-course counts. Remaining questions concern bounded detail such as exact temporal boundaries, geometry, operational use or historical terminology and are retained explicitly rather than converted into false precision.

## Sources / provenance

In [ ]:
provenance_tables = []
for path in racecourse_notebooks:
    notebook = json.loads(path.read_text())
    for cell in notebook.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        source = ''.join(cell.get('source', []))
        try:
            names = assigned_names(source)
        except SyntaxError:
            continue
        provenance_names = {name for name in names if 'provenance' in name.lower()}
        if not provenance_names:
            continue
        namespace = {'pd': pd}
        try:
            exec(source, namespace)
        except Exception:
            continue
        for name in sorted(provenance_names):
            value = namespace.get(name)
            if isinstance(value, pd.DataFrame) and len(value):
                table = value.copy()
                table['provenance_table'] = name
                table['source_notebook'] = path.name
                provenance_tables.append(table)

study03_provenance = pd.concat(provenance_tables, ignore_index=True, sort=False)
required = ['source_authority', 'source_title', 'source_url', 'accessed_date']
for column in required:
    assert column in study03_provenance.columns
    assert study03_provenance[column].fillna('').astype(str).str.strip().ne('').all(), f'Missing {column}'

study03_sources = (
    study03_provenance[required].drop_duplicates()
    .sort_values(['source_authority', 'source_title', 'source_url']).reset_index(drop=True)
)
print('assertion-level provenance records:', len(study03_provenance))
print('bibliographic source records:', len(study03_sources))
print('unique source URLs:', study03_sources['source_url'].nunique())
print('source notebooks:', study03_provenance['source_notebook'].nunique())
assert study03_provenance['source_notebook'].nunique() == 60

## Conclusion

A British **racecourse** is best treated as the recognised racing venue or institutional racecourse identity. It is not necessarily one physical racing course. A racecourse may contain one or several stable **course/track identities**, and those may in turn contain named routes/configurations or characteristics that change through time.

The 60 Study 03 evidence notebooks resolve the 65 British source labels to **61 governed racecourse identities**. The difference between notebook count and racecourse count is Newmarket: the Jockey Club officially treats the **Rowley Mile** and **July Course** as two racecourses, and the source labels allow them to be resolved separately.

Those 61 racecourses produce **90 course/track inventory records** and **86 stable course/track identities** at the Study 03 modelling level. **19 racecourses have more than one stable course/track identity.** The 86 is a governed inventory at this modelling level, not a claim that Britain has exactly 86 named physical route distinctions of every possible granularity.

The modelling implication is:

`racecourse -> course/track -> time-bounded characteristics`

A lower route/configuration layer can be added where later analysis needs it. `candidate_course_label` remains source-derived evidence and should not itself be mistaken for either racecourse identity or physical track identity.

**Bottom line: a British racecourse is the recognised racecourse/venue identity; it may contain multiple racing courses, and source labels are not a safe racecourse count.**

In [ ]:
assert source_label_mapping['racecourse_identity'].nunique() == 61
assert source_label_mapping['candidate_course_label'].nunique() == 65
assert len(course_inventory) == 90
assert len(stable_identities) == 86
assert (identity_counts['stable_course_identities'] > 1).sum() == 19
assert len(study03_unresolved) == 7
assert study03_provenance['source_notebook'].nunique() == 60
print('Study 03 national consolidation checks passed: 60 notebooks, 65 source labels, 61 racecourses, 86 stable course/track identities.')